# 02 — Preprocesamiento de transacciones

Lectura eficiente de `transactions.csv.gz`, validación de cada transacción contra la oferta de su cliente, decisiones de limpieza y construcción de las variables de comportamiento previo a la oferta. El resultado es `data/processed/dataset_modelo.csv`, una fila por cliente de train con sus variables y `repeater`, que es el insumo de los notebooks 03 a 06.

La lógica pesada vive en `src/preprocesamiento.py`; este notebook la ejecuta y documenta cada decisión.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from config import DATASET_MODELO_PATH, PROCESSED_DATA_DIR, TABLES_DIR
from preprocesamiento import (
    COLUMNAS_GENERALES,
    COLUMNAS_OFERTA,
    TAMANO_BLOQUE,
    cargar_clientes,
    comparar_memoria_dtypes,
    procesar_transacciones,
)

TABLES_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", None)

## 1. Clientes y su oferta

`transactions` no trae la columna `offer`. Cada transacción se relaciona con su cliente por `(id, chain)` y con el producto ofertado por `(category, company, brand)`. Por eso primero se arma una tabla con los clientes de train y test y el producto de la oferta que recibió cada uno.

In [2]:
clientes = cargar_clientes()
print("clientes:", clientes.shape)
print("ids repetidos entre train y test:", clientes.index.duplicated().sum())
clientes["split"].value_counts()

clientes: (311541, 12)
ids repetidos entre train y test: 0


split
train    160057
test     151484
Name: count, dtype: int64

In [3]:
clientes.head()

,chain,offer,market,repeattrips,repeater,offerdate,split,category,quantity,company,offervalue,brand
id,,,,,,,,,,,,
86246,205,1208251,34,5.0,True,2013-04-24,train,2202,1,104460040,2.00,3718
86252,205,1197502,34,16.0,True,2013-03-27,train,3203,1,106414464,0.75,13474
12682470,18,1197502,11,0.0,False,2013-03-28,train,3203,1,106414464,0.75,13474
12996040,15,1197502,9,0.0,False,2013-03-25,train,3203,1,106414464,0.75,13474
13089312,15,1204821,9,0.0,False,2013-04-01,train,5619,1,107717272,1.50,102504


## 2. Carga eficiente de `transactions.csv.gz`

El archivo pesa 2.9 GB comprimido y cerca de 22 GB descomprimido, así que no se puede cargar entero en memoria. Se aplican dos medidas:

1. Leerlo por bloques directamente desde el gzip con `chunksize`, sin descomprimirlo a disco.
2. Declarar tipos de dato más pequeños: enteros de 16 y 32 bits para los identificadores que caben en ellos, `float32` para `productsize` y fechas como `datetime`.

La tabla compara la memoria de un millón de filas con los tipos que pandas infiere por defecto contra los tipos reducidos.

In [4]:
memoria = comparar_memoria_dtypes()
memoria.to_csv(TABLES_DIR / "02_memoria_dtypes.csv")
memoria

,dtype_por_defecto,mb_por_defecto,dtype_reducido,mb_reducido
id,int64,8.0,int64,8.0
chain,int64,8.0,int16,2.0
dept,int64,8.0,int16,2.0
category,int64,8.0,int16,2.0
company,int64,8.0,int64,8.0
brand,int64,8.0,int32,4.0
date,str,18.0,datetime64[us],8.0
productsize,float64,8.0,float32,4.0
productmeasure,str,10.1,str,10.1
purchasequantity,int64,8.0,int32,4.0


Con los tipos reducidos un millón de filas pasa de 100 MB a 60 MB, un 40% menos. Aun así el archivo completo ocuparía unos 21 GB en memoria, por eso además se lee por bloques y nunca hay más de un bloque cargado a la vez.

## 3. Pasada por bloques

`procesar_transacciones()` recorre el archivo una sola vez, en bloques de 2 millones de filas, y en cada bloque:

1. **Contigüidad.** Las filas de cada cliente vienen juntas y ordenadas por `id`. Las filas del último cliente de cada bloque se retienen y se pasan al bloque siguiente, así cada cliente se resume completo una sola vez. La pasada cuenta los clientes que reaparecen en un bloque posterior (`clientes_no_contiguos`) y se detiene con error si hay alguno.
2. **Validación.** Cada fila se cruza con su cliente y se conserva solo si la cadena coincide con la de la oferta y la fecha es anterior a `offerdate`. Así ninguna variable usa información posterior a la oferta.
3. **Reducción.** Se marcan las filas cuya categoría o compañía aparece en `offers.csv`, el procedimiento que usaron los participantes de la competencia (Triskelion, 2014). Se guardan en `transacciones_reducidas.parquet`, que no se sube a git, para las variables avanzadas del notebook 06.
4. **Variables por cliente.** Gasto, compras, visitas, recencia, diversidad y compras previas de la categoría, compañía, marca y producto ofertados en ventanas de 30, 60, 90 y 180 días.
5. **Calidad y muestra.** Conteos de negativos, ceros y faltantes, estadísticas exactas de las variables crudas, totales por mes y una muestra aleatoria del 0.1% de las filas para los gráficos del notebook 03.

La pasada completa tarda unos minutos. Si las salidas ya existen se cargan de disco; para recalcular se usa `procesar_transacciones(forzar=True)`.

In [5]:
resultados = procesar_transacciones()
conteos = resultados["conteos"]
conteos.to_frame()

Las salidas ya existen: se cargan de disco. Usar forzar=True para recalcular.


,valor
concepto,
filas_leidas,349655789
filas_sin_cliente,0
filas_otra_cadena,113363
filas_en_o_despues_oferta,0
filas_validas,349542426
filas_reducidas,27756831
filas_devolucion,273848
filas_descuento,7847034
filas_monto_cero,5422153


## 4. Validación y reducción

In [6]:
total = conteos["filas_leidas"]
embudo = pd.DataFrame(
    {
        "filas": [total, conteos["filas_validas"], conteos["filas_reducidas"]],
    },
    index=[
        "leidas",
        "validas: cliente, cadena y fecha anterior a offerdate",
        "reducidas: categoria o compania de alguna oferta",
    ],
)
embudo["porcentaje"] = (100 * embudo["filas"] / total).round(2)
embudo.to_csv(TABLES_DIR / "02_embudo_filas.csv")
embudo

,filas,porcentaje
leidas,349655789,100.00
"validas: cliente, cadena y fecha anterior a offerdate",349542426,99.97
reducidas: categoria o compania de alguna oferta,27756831,7.94


In [7]:
descartes = conteos[["filas_sin_cliente", "filas_otra_cadena", "filas_en_o_despues_oferta", "clientes_no_contiguos"]]
clientes_cubiertos = conteos[["clientes_con_transacciones", "clientes_sin_transacciones"]]
pd.concat([descartes, clientes_cubiertos]).to_frame()

,valor
concepto,
filas_sin_cliente,0
filas_otra_cadena,113363
filas_en_o_despues_oferta,0
clientes_no_contiguos,0
clientes_con_transacciones,311541
clientes_sin_transacciones,0


**Resultado de la pasada**

- Se leyeron 349,655,789 filas en 175 bloques, en unos cuatro minutos y medio.
- Las filas de cada cliente vinieron contiguas: `clientes_no_contiguos` es 0, así que cada cliente se resumió completo y el dataset es exacto.
- Todas las transacciones son de clientes de train o test, y ninguna tiene fecha igual o posterior a la oferta del cliente. El historial ya viene cortado antes del cupón, así que ninguna variable puede usar información del futuro.
- 113,363 filas, el 0.03%, son del mismo cliente pero en otra cadena. Se descartan porque el reto relaciona transacciones y ofertas por `(id, chain)`.
- Los 311,541 clientes tienen historial; ninguno queda sin transacciones.
- La reducción a las categorías o compañías ofertadas deja 27,756,831 filas, el 7.9% del archivo, en línea con los cerca de 27 millones que reportaron los participantes de la competencia.

## 5. Negativos, ceros y faltantes en las transacciones

In [8]:
estadisticas = resultados["estadisticas"].set_index("variable")
estadisticas

,n,nulos,media,desviacion,min,max,negativos,ceros,suma
variable,,,,,,,,,
purchaseamount,349655789,0,4.489204,878.933674,-8593791.0,58658.76,8115002,5422153,1.569676e+09
purchasequantity,349655789,0,1.671760,40.313595,-32255.0,54800.00,273848,518736,5.845405e+08
productsize,349655789,0,27.325410,51.607789,0.0,6000.00,0,11507837,9.554488e+09


In [9]:
calidad = pd.DataFrame(
    {
        "filas": conteos[
            [
                "filas_devolucion",
                "filas_descuento",
                "filas_monto_cero",
                "filas_cantidad_cero",
                "filas_productsize_cero_o_nulo",
                "filas_productmeasure_nulo",
            ]
        ]
    }
)
calidad["porcentaje"] = (100 * calidad["filas"] / total).round(3)
calidad.to_csv(TABLES_DIR / "02_calidad_transacciones.csv")
calidad

,filas,porcentaje
concepto,,
filas_devolucion,273848,0.078
filas_descuento,7847034,2.244
filas_monto_cero,5422153,1.551
filas_cantidad_cero,518736,0.148
filas_productsize_cero_o_nulo,11507837,3.291
filas_productmeasure_nulo,11500472,3.289


Líneas de descuento, es decir con monto negativo y cantidad mayor o igual a cero, agrupadas por departamento y categoría. `categoria_ofertada` indica si la categoría es la de alguna oferta.

In [10]:
descuentos = resultados["descuentos"]
descuentos.head(10)

,dept,category,n_lineas,proporcion,categoria_ofertada
0,96,9609,5066677,0.645681,False
1,0,0,1137755,0.144992,False
2,97,9781,611408,0.077916,False
3,97,9753,72009,0.009177,False
4,9,907,46516,0.005928,False
5,63,6315,18521,0.002360,False
6,41,4107,16444,0.002096,False
7,4,416,15801,0.002014,False
8,5,501,14937,0.001904,False
9,63,6320,14548,0.001854,False


In [11]:
resultados["productmeasure"]

,productmeasure,n_lineas,proporcion
0,OZ,285018341,8.151398e-01
1,CT,42023867,1.201864e-01
2,SIN_MEDIDA,11500472,3.289084e-02
3,RL,4376413,1.251635e-02
4,LB,3927998,1.123390e-02
5,LT,2676799,7.655526e-03
6,YD,129309,3.698180e-04
7,1,1662,4.753246e-06
8,FT,524,1.498617e-06
9,QT,215,6.148904e-07


**Hallazgos y decisiones de limpieza**

- **Devoluciones.** Solo 273,848 líneas, el 0.08%, tienen cantidad negativa.
- **Líneas de descuento.** 7,847,034 líneas, el 2.2%, tienen monto negativo con cantidad positiva o cero. No son devoluciones: el 64.6% está en la categoría 9609 del departamento 96, con montos redondos y sin tamaño de producto, un 14.5% en la categoría 0 y un 7.8% en la 9781. Ninguna es una categoría ofertada. Todo apunta a cupones o descuentos registrados como una línea aparte.
- **Decisión sobre los negativos.** No se elimina ninguna fila. `gasto_total` es la suma neta, lo que el cliente pagó en realidad. Las devoluciones y los descuentos se cuentan en `n_devoluciones` y `n_descuentos`, y no cuentan como compra en `n_compras` ni en las compras de la categoría, compañía o marca ofertada.
- **Registros imposibles.** El mínimo de `purchaseamount` es -8,593,791 dólares, el de `purchasequantity` -32,255 unidades y el máximo de cantidad 54,800. Son errores de registro aislados: en la muestra del 0.1% solo 10 líneas superan los 1,000 dólares en valor absoluto. Por ellos la desviación del monto es 879 con una media de 4.49. Se conservan porque no hay forma de corregirlos sin inventar un valor; el notebook 03 los trata como atípicos y usa medidas robustas como la mediana y los percentiles.
- **Ceros.** 5,422,153 líneas con monto cero y 518,736 con cantidad cero. Se conservan en el historial pero no cuentan como compra.
- **Tamaño de producto.** `productsize` no tiene nulos, pero vale 0 en 11,507,837 líneas, el 3.3%, que son prácticamente las mismas 11,500,472 líneas sin `productmeasure`. Ese cero es un faltante disfrazado. Como ninguna variable del dataset usa el tamaño, no se imputa; solo se excluye de los gráficos de tamaño.
- **Unidades de medida.** El 81.5% de las líneas está en onzas y el 12.0% en conteo de unidades. Los tamaños solo son comparables dentro de una misma unidad.

## 6. Dataset por cliente

In [12]:
ds = resultados["dataset"]
ds_test = resultados["dataset_test"]
print("dataset_modelo (train):", ds.shape)
print("dataset_modelo_test:", ds_test.shape)
print("proporcion de repeater = True:", round(ds["repeater"].mean(), 4))
ds.head()

dataset_modelo (train): (160057, 49)
dataset_modelo_test: (151484, 47)
proporcion de repeater = True: 0.2714


,id,chain,market,offer,offerdate,category,company,brand,offervalue,quantity,repeattrips,repeater,n_lineas,n_compras,n_devoluciones,n_descuentos,n_visitas,gasto_total,ticket_promedio,dias_ultima_compra,antiguedad_dias,n_categorias_distintas,n_marcas_distintas,n_companias_distintas,n_compras_categoria,gasto_categoria,n_compras_categoria_30d,n_compras_categoria_60d,n_compras_categoria_90d,n_compras_categoria_180d,n_compras_compania,gasto_compania,n_compras_compania_30d,n_compras_compania_60d,n_compras_compania_90d,n_compras_compania_180d,n_compras_marca,gasto_marca,n_compras_marca_30d,n_compras_marca_60d,n_compras_marca_90d,n_compras_marca_180d,n_compras_producto,gasto_producto,dias_ultima_compra_categoria,nunca_compro_categoria,nunca_compro_compania,nunca_compro_marca,nunca_compro_producto
0,86246,205,34,1208251,2013-04-24,2202,104460040,3718,2.00,1,5,True,12609,12547,0,33,381,52828.12,138.66,1,418,599,1319,954,0,0.00,0,0,0,0,36,243.63,6,16,17,19,8,28.71,1,5,6,7,0,0.00,NaN,1,0,0,1
1,86252,205,34,1197502,2013-03-27,3203,106414464,13474,0.75,1,16,True,12087,12027,2,35,368,53592.90,145.63,1,390,590,1241,898,6,14.55,3,3,3,4,21,83.93,11,11,12,16,2,4.98,1,1,1,1,2,4.98,10.0,0,0,0,0
2,12682470,18,11,1197502,2013-03-28,3203,106414464,13474,0.75,1,0,False,806,756,0,41,141,3383.84,24.00,7,391,208,237,218,1,2.50,0,0,0,1,0,0.00,0,0,0,0,0,0.00,0,0,0,0,0,0.00,117.0,0,1,1,1
3,12996040,15,9,1197502,2013-03-25,3203,106414464,13474,0.75,1,0,False,326,321,0,5,39,1541.36,39.52,5,387,129,111,105,0,0.00,0,0,0,0,0,0.00,0,0,0,0,0,0.00,0,0,0,0,0,0.00,NaN,1,1,1,1
4,13089312,15,9,1204821,2013-04-01,5619,107717272,102504,1.50,1,0,False,1218,1160,0,52,132,3890.68,29.47,1,392,217,220,200,0,0.00,0,0,0,0,3,19.95,1,3,3,3,3,19.95,1,3,3,3,0,0.00,NaN,1,0,0,1


In [13]:
ds.dtypes.rename("dtype").to_frame().T

,id,chain,market,offer,offerdate,category,company,brand,offervalue,quantity,repeattrips,repeater,n_lineas,n_compras,n_devoluciones,n_descuentos,n_visitas,gasto_total,ticket_promedio,dias_ultima_compra,antiguedad_dias,n_categorias_distintas,n_marcas_distintas,n_companias_distintas,n_compras_categoria,gasto_categoria,n_compras_categoria_30d,n_compras_categoria_60d,n_compras_categoria_90d,n_compras_categoria_180d,n_compras_compania,gasto_compania,n_compras_compania_30d,n_compras_compania_60d,n_compras_compania_90d,n_compras_compania_180d,n_compras_marca,gasto_marca,n_compras_marca_30d,n_compras_marca_60d,n_compras_marca_90d,n_compras_marca_180d,n_compras_producto,gasto_producto,dias_ultima_compra_categoria,nunca_compro_categoria,nunca_compro_compania,nunca_compro_marca,nunca_compro_producto
dtype,int64,int64,int64,int64,datetime64[us],int64,int64,int64,float64,int64,int64,bool,int64,int64,int64,int64,int64,float64,float64,int64,int64,int64,int64,int64,int64,float64,int64,int64,int64,int64,int64,float64,int64,int64,int64,int64,int64,float64,int64,int64,int64,int64,int64,float64,float64,int64,int64,int64,int64


Diccionario de las variables construidas. Se exporta para que los notebooks 03 a 06 y el codebook usen las mismas definiciones.

In [14]:
descripciones = {
    "n_lineas": "lineas de transaccion en el historial, incluidas devoluciones y descuentos",
    "n_compras": "lineas de compra real: cantidad mayor a 0 y monto mayor a 0",
    "n_devoluciones": "lineas con cantidad negativa",
    "n_descuentos": "lineas con monto negativo y cantidad mayor o igual a 0",
    "n_visitas": "dias distintos con al menos una transaccion",
    "gasto_total": "suma neta de purchaseamount en dolares: compras menos devoluciones y descuentos",
    "ticket_promedio": "gasto_total dividido entre n_visitas",
    "dias_ultima_compra": "dias entre la ultima transaccion y offerdate",
    "antiguedad_dias": "dias entre la primera transaccion registrada y offerdate",
    "n_categorias_distintas": "categorias distintas compradas",
    "n_marcas_distintas": "marcas distintas compradas",
    "n_companias_distintas": "companias distintas compradas",
    "n_compras_producto": "compras previas del mismo producto: misma categoria, compania y marca de la oferta",
    "gasto_producto": "monto neto gastado en el mismo producto de la oferta",
    "dias_ultima_compra_categoria": "dias desde la ultima compra de la categoria ofertada; vacio si nunca la compro",
}
nombres_dim = {"categoria": "la categoria", "compania": "la compania", "marca": "la marca"}
for dim, texto in nombres_dim.items():
    descripciones[f"n_compras_{dim}"] = f"compras previas de {texto} ofertada"
    descripciones[f"gasto_{dim}"] = f"monto neto gastado en {texto} ofertada"
    for dias in (30, 60, 90, 180):
        descripciones[f"n_compras_{dim}_{dias}d"] = f"compras de {texto} ofertada en los {dias} dias previos a offerdate"
for dim, texto in {**nombres_dim, "producto": "el producto"}.items():
    descripciones[f"nunca_compro_{dim}"] = f"1 si no hay compras previas de {texto} ofertada"

diccionario = pd.DataFrame(
    {"variable": COLUMNAS_GENERALES + COLUMNAS_OFERTA}
).assign(descripcion=lambda d: d["variable"].map(descripciones))
assert diccionario["descripcion"].notna().all()
diccionario.to_csv(TABLES_DIR / "02_diccionario_dataset_modelo.csv", index=False)
diccionario

,variable,descripcion
0,n_lineas,"lineas de transaccion en el historial, incluid..."
1,n_compras,lineas de compra real: cantidad mayor a 0 y mo...
2,n_devoluciones,lineas con cantidad negativa
3,n_descuentos,lineas con monto negativo y cantidad mayor o i...
4,n_visitas,dias distintos con al menos una transaccion
5,gasto_total,suma neta de purchaseamount en dolares: compra...
6,ticket_promedio,gasto_total dividido entre n_visitas
7,dias_ultima_compra,dias entre la ultima transaccion y offerdate
8,antiguedad_dias,dias entre la primera transaccion registrada y...
9,n_categorias_distintas,categorias distintas compradas


## 7. Faltantes en el dataset

In [15]:
faltantes = ds.isna().sum().rename("faltantes").to_frame()
faltantes["porcentaje"] = (100 * faltantes["faltantes"] / len(ds)).round(2)
faltantes = faltantes[faltantes["faltantes"] > 0]
faltantes.to_csv(TABLES_DIR / "02_faltantes_dataset.csv")
faltantes

,faltantes,porcentaje
dias_ultima_compra_categoria,72639,45.38


In [16]:
sin_historial = ds[[c for c in ds.columns if c.startswith("nunca_compro_")]].mean().rename("proporcion_clientes")
sin_historial.to_frame()

,proporcion_clientes
nunca_compro_categoria,0.453832
nunca_compro_compania,0.458980
nunca_compro_marca,0.606034
nunca_compro_producto,0.843043


**Decisión sobre los faltantes**

Solo `dias_ultima_compra_categoria` tiene vacíos: 72,639 clientes, el 45.38%, que son exactamente los que nunca compraron la categoría ofertada. No es un dato perdido sino un evento que no ocurrió. No se imputa con la media, que inventaría una compra, ni con cero, que significaría una compra el día anterior. La información queda en la bandera `nunca_compro_categoria`. Para los modelos del notebook 05 se puede reemplazar por un valor mayor que el historial máximo o usar un modelo que acepte vacíos.

En los conteos y montos el cero sí es un valor real y por eso se completaron con 0. La falta de historial del producto es frecuente: el 45.4% nunca compró la categoría ofertada, el 45.9% la compañía, el 60.6% la marca y el 84.3% el producto exacto.

## 8. Archivos generados

In [17]:
salidas = sorted(PROCESSED_DATA_DIR.glob("*")) + sorted(TABLES_DIR.glob("02_*.csv"))
pd.DataFrame(
    {"archivo": [str(p.relative_to(PROCESSED_DATA_DIR.parent.parent)) for p in salidas],
     "mb": [round(p.stat().st_size / 1e6, 2) for p in salidas]}
)

,archivo,mb
0,data/processed/dataset_modelo.csv,28.84
1,data/processed/dataset_modelo_test.csv,26.29
2,data/processed/train_offers.csv,11.29
3,data/processed/transacciones_muestra.csv,21.84
4,data/processed/transacciones_reducidas.parquet,233.49
5,results/tables/02_calidad_transacciones.csv,0.00
6,results/tables/02_conteos_limpieza.csv,0.00
7,results/tables/02_diccionario_dataset_modelo.csv,0.00
8,results/tables/02_embudo_filas.csv,0.00
9,results/tables/02_estadisticas_transacciones.csv,0.00


`dataset_modelo.csv`, `dataset_modelo_test.csv` y la muestra pesan menos de 30 MB cada uno y se suben al repositorio, para que los notebooks 03 a 06 funcionen sin repetir la pasada. `transacciones_reducidas.parquet` está en `.gitignore`; quien lo necesite para el notebook 06 lo regenera con `procesar_transacciones(forzar=True)`.